# Proyecto Final — Módulo Pandas
## Caso: Campaña Snacks × Mundial FIFA 2026 — Guatemala

### Justificación del esquema
Se utilizan dos librerías complementarias: **`csv`** para auditar el archivo raw sin inferencia de tipos, y **`pandas`** para la limpieza, transformación y análisis. Este flujo de dos pasadas es buena práctica ETL: detectar problemas primero, transformar después.

## Fase 1 — Carga de Datos

In [ ]:
import csv
import pandas as pd

RUTA_CSV = "encuesta_snacks_mundial_2026_guatemala_2500_respuestas.csv"

In [ ]:
# Auditoría con csv — sin inferencia de tipos
filas_raw = []
with open(RUTA_CSV, encoding="utf-8-sig") as f:
    lector = csv.DictReader(f)
    columnas = lector.fieldnames
    for fila in lector:
        filas_raw.append(fila)

print(f"Columnas  : {len(columnas)}")
print(f"Registros : {len(filas_raw)}")

print("\nVacíos por columna:")
for col in columnas:
    vacios = sum(1 for f in filas_raw if not f[col].strip())
    if vacios > 0:
        print(f"  {col}: {vacios}")

conteo_ids = {}
for f in filas_raw:
    conteo_ids[f["EncuestaID"]] = conteo_ids.get(f["EncuestaID"], 0) + 1
dups = [k for k, v in conteo_ids.items() if v > 1]
print(f"\nIDs duplicados: {len(dups)}")

In [ ]:
# Carga en pandas — todo como string para control manual de tipos
df = pd.read_csv(RUTA_CSV, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"Shape: {df.shape}")
df.head(3)

## Fase 2 — Exploración Inicial

In [ ]:
print("Valores únicos por columna:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")

In [ ]:
# Distribución de columnas clave
for col in ["RangoEdad", "Genero", "FrecuenciaConsumoSnacks", "GastoSnacksPartido", "PrecioAdecuado"]:
    print(f"\n── {col}")
    conteo = {}
    for val in df[col]:
        v = val.strip() if val.strip() else "(vacío)"
        conteo[v] = conteo.get(v, 0) + 1
    for k, v in sorted(conteo.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v}")

fechas = sorted(df["FechaEncuesta"].tolist())
print(f"\nRango de fechas: {fechas[0]} → {fechas[-1]}")

## Fase 3 — Limpieza de Datos

In [ ]:
df_limpio = df.copy()
n_original = len(df_limpio)

# Strip general
for col in df_limpio.columns:
    df_limpio[col] = df_limpio[col].str.strip()

# Eliminar duplicados — se conserva la primera ocurrencia
df_limpio = df_limpio.drop_duplicates(subset="EncuestaID", keep="first")
print(f"Duplicados eliminados : {n_original - len(df_limpio)}")

# Normalizar SaborPreferido — inconsistencias: 'QUESO', 'Queso', 'nan' como texto
df_limpio["SaborPreferido"] = df_limpio["SaborPreferido"].replace("nan", "")
df_limpio["SaborPreferido"] = df_limpio["SaborPreferido"].str.title()

# Imputar vacíos con 'No especificado' en columnas de texto libre
for col in ["Municipio", "Ocupacion", "LugarCompraSnacks", "SeleccionInfluyeCompra"]:
    df_limpio[col] = df_limpio[col].replace("", "No especificado")

# Imputar vacíos con la moda en columnas categóricas
for col in ["SaborPreferido", "PrecioAdecuado"]:
    moda = df_limpio[df_limpio[col] != ""][col].value_counts().idxmax()
    df_limpio[col] = df_limpio[col].replace("", moda)
    print(f"  {col}: imputado con moda '{moda}'")

# Convertir tipos
df_limpio["EncuestaID"]    = df_limpio["EncuestaID"].astype(int)
df_limpio["FechaEncuesta"] = pd.to_datetime(df_limpio["FechaEncuesta"], format="%Y-%m-%d", errors="coerce")
df_limpio["HoraEncuesta"]  = pd.to_datetime(df_limpio["HoraEncuesta"], format="%H:%M:%S", errors="coerce").dt.time

print(f"\nRegistros limpios: {len(df_limpio)}")

## Fase 4 — Transformación

In [ ]:
# Segmento de edad
mapa_segmento = {
    "Menos de 18 años": "Juvenil",
    "18 - 24 años"    : "Joven Adulto",
    "25 - 34 años"    : "Adulto Joven",
    "35 - 44 años"    : "Adulto",
    "45 - 54 años"    : "Adulto Mayor",
    "55 años o más"   : "Senior"
}
df_limpio["SegmentoEdad"] = df_limpio["RangoEdad"].map(mapa_segmento).fillna("No especificado")

# Categoría de precio
mapa_precio = {
    "Menos de Q10": "Economico",
    "Q10 - Q15"   : "Accesible",
    "Q16 - Q20"   : "Moderado",
    "Q21 - Q30"   : "Premium",
    "Más de Q30"  : "Super Premium"
}
df_limpio["CategoriaPrecio"] = df_limpio["PrecioAdecuado"].map(mapa_precio).fillna("No especificado")

# Cantidad de snacks por encuestado (columna multi-valor separada por ';')
df_limpio["CantidadSnacks"] = df_limpio["SnacksSeleccionados"].apply(
    lambda x: len([s for s in x.split(";") if s.strip()])
)

# Alta intención de compra — 1 si pagaría más por edición Mundial
df_limpio["AltaIntencionCompra"] = df_limpio["PagaMasEdicionMundial"].apply(
    lambda x: 1 if x.lower() == "sí" else 0
)

# Mes de encuesta
df_limpio["MesEncuesta"] = df_limpio["FechaEncuesta"].dt.month

print("Columnas derivadas creadas:")
for col in ["SegmentoEdad", "CategoriaPrecio", "CantidadSnacks", "AltaIntencionCompra", "MesEncuesta"]:
    print(f"  + {col}")

## Fase 5 — Star Schema
```
fact_encuesta
    ├── dim_encuestado
    ├── dim_ubicacion
    ├── dim_tiempo
    ├── dim_snack
    └── dim_campania
```

In [ ]:
# dim_encuestado — perfil demográfico
dim_encuestado = df_limpio[["EncuestaID", "RangoEdad", "SegmentoEdad", "Genero", "Ocupacion"]].copy()
dim_encuestado = dim_encuestado.rename(columns={"EncuestaID": "id_encuestado"})
dim_encuestado = dim_encuestado.drop_duplicates(subset="id_encuestado").reset_index(drop=True)

# dim_ubicacion — ubicación geográfica
dim_ubicacion = df_limpio[["EncuestaID", "Departamento", "Municipio"]].copy()
dim_ubicacion = dim_ubicacion.rename(columns={"EncuestaID": "id_ubicacion"})
dim_ubicacion = dim_ubicacion.drop_duplicates(subset="id_ubicacion").reset_index(drop=True)

# dim_tiempo — información temporal
dim_tiempo = df_limpio[["EncuestaID", "FechaEncuesta", "MesEncuesta", "HoraEncuesta"]].copy()
dim_tiempo = dim_tiempo.rename(columns={"EncuestaID": "id_tiempo"})
dim_tiempo = dim_tiempo.drop_duplicates(subset="id_tiempo").reset_index(drop=True)

# dim_snack — un snack por fila (explode de columna multi-valor)
dim_snack = df_limpio[["EncuestaID", "SnacksSeleccionados"]].copy()
dim_snack["Snack"] = dim_snack["SnacksSeleccionados"].str.split(";")
dim_snack = dim_snack.explode("Snack")
dim_snack["Snack"] = dim_snack["Snack"].str.strip()
dim_snack = dim_snack[dim_snack["Snack"] != ""].reset_index(drop=True)
dim_snack = dim_snack.drop(columns=["SnacksSeleccionados"])
dim_snack = dim_snack.rename(columns={"EncuestaID": "id_encuestado"})

# dim_campania — preferencias publicitarias
dim_campania = df_limpio[["EncuestaID", "TipoPublicidadAtractiva", "PromocionPreferida",
                           "CampaniaMasProbableCompra", "CompraTarjetasColeccionables"]].copy()
dim_campania = dim_campania.rename(columns={"EncuestaID": "id_campania"})
dim_campania = dim_campania.drop_duplicates(subset="id_campania").reset_index(drop=True)

# fact_encuesta — métricas y claves foráneas
fact_encuesta = df_limpio[["EncuestaID", "GastoSnacksPartido", "CantidadSnacks",
                            "AltaIntencionCompra", "CategoriaPrecio", "FrecuenciaConsumoSnacks",
                            "SeleccionApoya", "JugadoresInfluyentes",
                            "PlaneaVerMundial2026", "ConQuienVePartidos"]].copy()
fact_encuesta = fact_encuesta.rename(columns={"EncuestaID": "id_encuesta"})
fact_encuesta["id_encuestado"] = fact_encuesta["id_encuesta"]
fact_encuesta["id_ubicacion"]  = fact_encuesta["id_encuesta"]
fact_encuesta["id_tiempo"]     = fact_encuesta["id_encuesta"]
fact_encuesta["id_campania"]   = fact_encuesta["id_encuesta"]

for nombre, tabla in [("dim_encuestado", dim_encuestado), ("dim_ubicacion", dim_ubicacion),
                       ("dim_tiempo", dim_tiempo), ("dim_snack", dim_snack),
                       ("dim_campania", dim_campania), ("fact_encuesta", fact_encuesta)]:
    print(f"{nombre}: {tabla.shape}")

In [ ]:
# Exportar star schema
dim_encuestado.to_csv("dim_encuestado.csv", index=False, encoding="utf-8-sig")
dim_ubicacion.to_csv("dim_ubicacion.csv",   index=False, encoding="utf-8-sig")
dim_tiempo.to_csv("dim_tiempo.csv",         index=False, encoding="utf-8-sig")
dim_snack.to_csv("dim_snack.csv",           index=False, encoding="utf-8-sig")
dim_campania.to_csv("dim_campania.csv",     index=False, encoding="utf-8-sig")
fact_encuesta.to_csv("fact_encuesta.csv",   index=False, encoding="utf-8-sig")
df_limpio.to_csv("encuesta_limpia.csv",     index=False, encoding="utf-8-sig")
print("Archivos exportados exitosamente.")

## Fase 6 — Generación de Reportes

In [ ]:
# Reporte 1: Perfil Demográfico
reporte_demografico = pd.crosstab(
    df_limpio['SegmentoEdad'], df_limpio['Genero'], normalize='all'
) * 100
reporte_demografico = reporte_demografico.round(2)
reporte_demografico['Total Segmento (%)'] = reporte_demografico.sum(axis=1)
print("REPORTE 1 — Perfil Demográfico:\n")
print(reporte_demografico.sort_values(by='Total Segmento (%)', ascending=False))

In [ ]:
# Reporte 2: Frecuencia de Consumo
reporte_frecuencia = df_limpio['FrecuenciaConsumoSnacks'].value_counts().reset_index()
reporte_frecuencia.columns = ['Frecuencia', 'Cantidad de Clientes']
reporte_frecuencia['Porcentaje (%)'] = round((reporte_frecuencia['Cantidad de Clientes'] / len(df_limpio)) * 100, 2)
print("REPORTE 2 — Frecuencia de Consumo:\n")
print(reporte_frecuencia)

# Cruce: frecuencia vs intención de pago premium
frecuencia_vs_intencion = df_limpio.groupby('FrecuenciaConsumoSnacks')['AltaIntencionCompra'].mean().reset_index()
frecuencia_vs_intencion['AltaIntencionCompra'] = round(frecuencia_vs_intencion['AltaIntencionCompra'] * 100, 2)
frecuencia_vs_intencion.columns = ['Frecuencia', 'Probabilidad de Pagar Más (%)']
print("\nProbabilidad de pago premium por frecuencia:\n")
print(frecuencia_vs_intencion.sort_values(by='Probabilidad de Pagar Más (%)', ascending=False))

In [ ]:
# Reporte 3: Ranking de Snacks
ranking_snacks = dim_snack['Snack'].value_counts().reset_index()
ranking_snacks.columns = ['Tipo de Snack', 'Menciones']
ranking_snacks['Porcentaje de Preferencia (%)'] = round((ranking_snacks['Menciones'] / len(df_limpio)) * 100, 2)
print("Top 5 Snacks:\n")
print(ranking_snacks.head(5))

ranking_sabores = df_limpio['SaborPreferido'].value_counts().reset_index()
ranking_sabores.columns = ['Sabor', 'Preferencias']
ranking_sabores['Porcentaje (%)'] = round((ranking_sabores['Preferencias'] / len(df_limpio)) * 100, 2)
print("\nTop 5 Sabores:\n")
print(ranking_sabores.head(5))

In [ ]:
# Reporte 4: Análisis de Precios
reporte_precios = df_limpio['PrecioAdecuado'].value_counts().reset_index()
reporte_precios.columns = ['Rango de Precio', 'Cantidad']
reporte_precios['Porcentaje (%)'] = round((reporte_precios['Cantidad'] / len(df_limpio)) * 100, 2)
print("REPORTE 4 — Preferencia de Precios:\n")
print(reporte_precios)

In [ ]:
# Reporte 5: Equipos con Mayor Influencia
ranking_selecciones = df_limpio['SeleccionInfluyeCompra'].value_counts().reset_index()
ranking_selecciones.columns = ['Selección', 'Menciones']
ranking_selecciones['Porcentaje (%)'] = round((ranking_selecciones['Menciones'] / len(df_limpio)) * 100, 2)
print("Top 5 Selecciones con mayor influencia:\n")
print(ranking_selecciones.head(5))

In [ ]:
# Reporte 6: Jugadores con Mayor Influencia
jugadores_lista = df_limpio['JugadoresInfluyentes'].str.split(';').explode().str.strip()
ranking_jugadores = jugadores_lista[jugadores_lista != ""].value_counts().reset_index()
ranking_jugadores.columns = ['Jugador', 'Menciones']
ranking_jugadores['Porcentaje (%)'] = round((ranking_jugadores['Menciones'] / len(df_limpio)) * 100, 2)
print("Top 5 Jugadores:\n")
print(ranking_jugadores.head(5))

In [ ]:
# Reporte 7: Tipo de Publicidad más Efectiva
ranking_publicidad = df_limpio['TipoPublicidadAtractiva'].value_counts().reset_index()
ranking_publicidad.columns = ['Tipo Publicidad', 'Cantidad']
ranking_publicidad['Porcentaje (%)'] = round((ranking_publicidad['Cantidad'] / len(df_limpio)) * 100, 2)
print("REPORTE 7 — Publicidad más Efectiva:\n")
print(ranking_publicidad)

In [ ]:
# Reporte 8: Promociones Preferidas
ranking_promociones = df_limpio['PromocionPreferida'].value_counts().reset_index()
ranking_promociones.columns = ['Promoción', 'Cantidad']
ranking_promociones['Porcentaje (%)'] = round((ranking_promociones['Cantidad'] / len(df_limpio)) * 100, 2)
print("REPORTE 8 — Promociones Preferidas:\n")
print(ranking_promociones)

In [ ]:
# Reporte 9: Intención de Compra por Campaña
ranking_campania = df_limpio['CampaniaMasProbableCompra'].value_counts().reset_index()
ranking_campania.columns = ['Campaña', 'Cantidad']
ranking_campania['Porcentaje (%)'] = round((ranking_campania['Cantidad'] / len(df_limpio)) * 100, 2)
print("Campañas con mayor intención de compra:\n")
print(ranking_campania)

# Cruce: campaña vs alta intención de pago premium
campania_vs_intencion = df_limpio.groupby('CampaniaMasProbableCompra')['AltaIntencionCompra'].mean().reset_index()
campania_vs_intencion['AltaIntencionCompra'] = round(campania_vs_intencion['AltaIntencionCompra'] * 100, 2)
campania_vs_intencion.columns = ['Campaña', 'Clientes Dispuestos a Pagar Más (%)']
print("\nCampañas con mayor conversión a pago premium:\n")
print(campania_vs_intencion.sort_values(by='Clientes Dispuestos a Pagar Más (%)', ascending=False))

In [ ]:
# Reporte 10: Recomendación Estratégica Final
snack_top      = dim_snack['Snack'].value_counts().idxmax()
sabor_top      = df_limpio['SaborPreferido'].value_counts().idxmax()
segmento_top   = df_limpio['SegmentoEdad'].value_counts().idxmax()
seleccion_top  = df_limpio['SeleccionInfluyeCompra'].str.title().value_counts().idxmax()
jugador_top    = df_limpio['JugadoresInfluyentes'].str.split(';').explode().str.strip().value_counts().idxmax()
publicidad_top = df_limpio['TipoPublicidadAtractiva'].value_counts().idxmax()
promocion_top  = df_limpio['PromocionPreferida'].value_counts().idxmax()
precio_top     = df_limpio['PrecioAdecuado'].value_counts().idxmax()
campania_top   = df_limpio['CampaniaMasProbableCompra'].value_counts().idxmax()

print("PROPUESTA DE CAMPAÑA — SNACKS × MUNDIAL FIFA 2026")
print("=" * 55)
print(f"  Snack a promocionar    : {snack_top}")
print(f"  Sabor principal        : {sabor_top}")
print(f"  Segmento objetivo      : {segmento_top}")
print(f"  Selección influyente   : {seleccion_top}")
print(f"  Jugador embajador      : {jugador_top}")
print(f"  Canal publicitario     : {publicidad_top}")
print(f"  Promoción recomendada  : {promocion_top}")
print(f"  Precio ideal           : {precio_top}")
print(f"  Campaña de empaque     : {campania_top}")